# Click VB → DP → Bragg peaks → ACOM Si (fiber)

Flujo para indexar un DP de Si **sin asumir rotación in-plane = 0°**:

1. Cargar VB del `.h5` sidecar + cube 4D del `.mib` (memmap)
2. Click en un pixel `(ry, rx)` → extraer DP
3. Obtener Bragg peaks en ese pixel (preferir `*braggpeaks.h5`; si no, detectar)
4. Construir `Crystal` Si + `orientation_plan(fiber)` con ZA aproximado
5. `match_single_pattern` → overlay simulado con labels `(hkl)`

**Convención Fast4d:** click `x→rx`, `y→ry`; DP = `dc.data[ry, rx]`.

Requiere `ipympl` (`%matplotlib widget`) y py4DSTEM ≥ 0.14.

## 1. Imports + backend interactivo

In [ ]:
%matplotlib widget

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# repo root = parent of notebooks/
ROOT = Path.cwd()
if not (ROOT / "pipeline.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "pipeline.py").is_file(), f"No encuentro Fast4d en {ROOT}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import engine as E
import pipeline as pl
import py4DSTEM

print("ROOT", ROOT)
print("py4DSTEM", py4DSTEM.__version__)

## 2. Configuración

- `H5_PATH`: sidecar VB/ADF (ligero)
- `RAW_PATH`: `.mib` 4D (auto sibling si `None`)
- `BRAGG_PATH`: braggpeaks ya detectados (recomendado; auto `*braggpeaks.h5`)
- `ZA_APPROX` / `FIBER_TILT_DEG`: restricción suave del zone axis (no fija rotación del patrón)

In [ ]:
H5_PATH = Path(
    r"C:\Users\jtapiaca.ASURITE\Desktop\4D-STEM Data\YS-C\07202026 - Second inspection\4D\Scan05.h5"
)
RAW_PATH = None          # None → sibling .mib
BRAGG_PATH = None        # None → auto Scan05_braggpeaks.h5 / Scan05braggpeaks.h5
VB_KEY = "bright_field"  # o "annular_dark_field"
SCAN_SHAPE = None        # (R_Ny, R_Nx) solo si MIB metadata falla

# --- ACOM Si ---
ZA_APPROX = [0, 1, 1]    # zone axis aproximado (restricción fiber)
FIBER_TILT_DEG = 5.0     # cono de tilt permitido alrededor del ZA
ANGLE_STEP_IN_PLANE = 2.0
ANGLE_STEP_ZONE_AXIS = 1.0
ACCEL_VOLTAGE = 300e3
K_MAX = 1.5              # Å^-1
CORR_KERNEL_SIZE = 0.08  # Å^-1

# Si braggpeaks no están calibrados en Å^-1, usa estos fallbacks (px):
Q_PIXEL_AINV = None      # ej. 0.012; None → leer del h5 / bragg calibration
QX0_PX = None            # origen fila; None → centro del DP o cal
QY0_PX = None            # origen col

assert H5_PATH.is_file(), f"No existe: {H5_PATH}"

if RAW_PATH is None:
    cand = H5_PATH.with_suffix(".mib")
    RAW_PATH = cand if cand.is_file() else H5_PATH
RAW_PATH = Path(RAW_PATH)
assert RAW_PATH.is_file(), f"No existe raw/cube: {RAW_PATH}"

if BRAGG_PATH is None:
    for name in (
        f"{H5_PATH.stem}_braggpeaks.h5",
        f"{H5_PATH.stem}braggpeaks.h5",
        f"{RAW_PATH.stem}_braggpeaks.h5",
        f"{RAW_PATH.stem}braggpeaks.h5",
    ):
        cand = H5_PATH.with_name(name)
        if cand.is_file():
            BRAGG_PATH = cand
            break

print(f"H5 (VB) : {H5_PATH}")
print(f"RAW (4D): {RAW_PATH}  ({RAW_PATH.stat().st_size / 1e9:.1f} GB)")
print(f"BRAGG   : {BRAGG_PATH if BRAGG_PATH else '(none — se detectará en el pixel clickeado)'}")

## 3. Explorar árbol H5 (opcional)

In [ ]:
def print_tree(node, indent=0, max_depth=3):
    if node is None or indent > max_depth:
        return
    shape = f" {node['shape']}" if node.get("shape") else ""
    print("  " * indent + f"{node['name']} [{node['kind']}]{shape}")
    for ch in node.get("children", []) or []:
        print_tree(ch, indent + 1, max_depth)

print_tree(E.explore_h5(str(H5_PATH)))

## 4. Cargar VB + datacube 4D

El sidecar `.h5` típico **no** tiene el cube 4D; el `.mib` sí (memmap).

In [ ]:
vimgs = E._read_virtual_images_lenient(str(H5_PATH))
print("virtual images:", list(vimgs.keys()))

vb = vimgs.get(VB_KEY)
if vb is None:
    for alt in ("bright_field", "annular_dark_field"):
        if alt in vimgs:
            VB_KEY = alt
            vb = vimgs[alt]
            break
assert vb is not None and vb.ndim == 2, f"Sin VB 2D. Keys={list(vimgs)}"

R_Ny, R_Nx = map(int, vb.shape)
print(f"VB key={VB_KEY!r} shape=(R_Ny,R_Nx)=({R_Ny},{R_Nx})")

raw = Path(RAW_PATH)
if raw.suffix.lower() == ".mib":
    dc = pl._load_mib(
        raw,
        mem="memmap",
        scan=SCAN_SHAPE or (R_Ny, R_Nx),
        log=print,
    )
elif raw.suffix.lower() in (".h5", ".hdf5", ".emd"):
    dc = pl._load_emd_h5_datacube(raw, log=print)
else:
    dc = pl._load_raw_datacube(raw, log=print)

rshape = tuple(getattr(dc, "Rshape", ()) or ())
qshape = tuple(getattr(dc, "Qshape", ()) or ())
if len(rshape) == 2 and len(qshape) == 2:
    assert rshape == (R_Ny, R_Nx), f"Mismatch R: VB {vb.shape} vs cube {rshape}"
    Q_Ny, Q_Nx = map(int, qshape)
else:
    cube = np.asarray(getattr(dc, "data", dc))
    assert cube.ndim == 4 and cube.shape[:2] == (R_Ny, R_Nx)
    Q_Ny, Q_Nx = map(int, cube.shape[2:])

print(f"datacube R=({R_Ny},{R_Nx}) Q=({Q_Ny},{Q_Nx})")
dp_smoke = E._dp_at(dc, R_Ny // 2, R_Nx // 2)
print(f"smoke DP center shape={dp_smoke.shape} min/max={dp_smoke.min():.3g}/{dp_smoke.max():.3g}")

## 5. Click en VB → mostrar DP

Click izquierdo sobre la VB. El panel derecho muestra el DP; `last` guarda `(ry, rx, dp)`.

In [ ]:
VIEW_MODE = "highpass"
CMAP_VB = "gray"
CMAP_DP = "inferno"

fig, (ax_vb, ax_dp) = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)

ax_vb.imshow(vb, cmap=CMAP_VB, origin="upper")
ax_vb.set_title(f"{VB_KEY}  — click a pixel")
ax_vb.set_xlabel("rx (col)")
ax_vb.set_ylabel("ry (row)")
marker, = ax_vb.plot([], [], "r+", ms=14, mew=1.5)

ry0, rx0 = R_Ny // 2, R_Nx // 2
dp0 = E._dp_at(dc, ry0, rx0)
im_dp = ax_dp.imshow(E._view_filter(dp0, mode=VIEW_MODE), cmap=CMAP_DP, origin="upper")
ax_dp.set_title(f"DP @ (ry,rx)=({ry0},{rx0})")
ax_dp.set_xlabel("qx")
ax_dp.set_ylabel("qy")

last = {"ry": ry0, "rx": rx0, "dp": dp0}


def on_click(event):
    if event.inaxes is not ax_vb or event.xdata is None or event.ydata is None:
        return
    rx = int(round(event.xdata))
    ry = int(round(event.ydata))
    if not (0 <= ry < R_Ny and 0 <= rx < R_Nx):
        return
    dp = E._dp_at(dc, ry, rx)
    last.update(ry=ry, rx=rx, dp=dp)
    marker.set_data([rx], [ry])
    im_dp.set_data(E._view_filter(dp, mode=VIEW_MODE))
    im_dp.set_clim(0, 1)
    ax_dp.set_title(f"DP @ (ry,rx)=({ry},{rx})")
    fig.canvas.draw_idle()
    print(f"clicked ry={ry}, rx={rx}  dp.shape={dp.shape}  min/max={dp.min():.3g}/{dp.max():.3g}")


cid = fig.canvas.mpl_connect("button_press_event", on_click)
plt.show()

## 6. Extraer último DP

Corre esta celda **después** de clickear.

In [ ]:
ry, rx = int(last["ry"]), int(last["rx"])
dp = np.asarray(last["dp"], dtype=np.float32)
print(f"(ry, rx) = ({ry}, {rx})")
print(f"dp.shape = {dp.shape}, dtype={dp.dtype}, min/max={dp.min():.3g}/{dp.max():.3g}")

## 7. Bragg peaks en el pixel clickeado

Prioridad:
1. Cargar `BRAGG_PATH` y leer `bragg.cal[ry, rx]` (o `raw` + calibración local)
2. Si no hay braggpeaks: construir probe desde `dp_mean` / DP y detectar solo en `(ry, rx)`

In [ ]:
import h5py
from py4DSTEM import PointList


def _read_h5_scalar(path, *candidates):
    with h5py.File(path, "r") as f:
        for key in candidates:
            if key in f:
                return np.asarray(f[key][()]).reshape(-1)[0]
            # walk leaf name
            found = {}

            def visit(name, obj):
                if name.split("/")[-1] == key.split("/")[-1] and isinstance(obj, h5py.Dataset):
                    found["v"] = np.asarray(obj[()]).reshape(-1)[0]

            f.visititems(visit)
            if "v" in found:
                return found["v"]
    return None


def _pointlist_from_arrays(qx, qy, intensity=None):
    qx = np.asarray(qx, dtype=np.float64).ravel()
    qy = np.asarray(qy, dtype=np.float64).ravel()
    n = min(qx.size, qy.size)
    qx, qy = qx[:n], qy[:n]
    if intensity is None:
        intensity = np.ones(n, dtype=np.float64)
    else:
        intensity = np.asarray(intensity, dtype=np.float64).ravel()[:n]
    data = np.zeros(n, dtype=[("qx", "f8"), ("qy", "f8"), ("intensity", "f8")])
    data["qx"] = qx
    data["qy"] = qy
    data["intensity"] = intensity
    try:
        return PointList(data=data)
    except TypeError:
        return PointList(data)


def _peaks_to_pointlist(peaks):
    """Accept PointList / ndarray / structured array → PointList with qx,qy,intensity."""
    if peaks is None:
        raise ValueError("No peaks")
    # Already a usable PointList with qx/qy
    if hasattr(peaks, "data"):
        d = peaks.data
        names = set(getattr(peaks, "fields", ()) or (getattr(d.dtype, "names", None) or ()))
        if {"qx", "qy"}.issubset(names):
            # Keep as-is when possible (preserves calibration metadata on the object)
            if "intensity" in names and type(peaks).__name__ == "PointList":
                return peaks
            inten = d["intensity"] if "intensity" in names else np.ones(len(d["qx"]))
            return _pointlist_from_arrays(d["qx"], d["qy"], inten)
        if getattr(d, "ndim", 0) == 2 and d.shape[1] >= 2:
            inten = d[:, 2] if d.shape[1] >= 3 else None
            return _pointlist_from_arrays(d[:, 0], d[:, 1], inten)
    arr = np.asarray(peaks)
    if arr.ndim == 2 and arr.shape[1] >= 2:
        inten = arr[:, 2] if arr.shape[1] >= 3 else None
        return _pointlist_from_arrays(arr[:, 0], arr[:, 1], inten)
    raise TypeError(f"Cannot parse peaks type={type(peaks)}")


def _calibrate_peaks_px_to_ainv(peaks_px, qx0, qy0, q_pixel):
    """Detector px → centered Å^-1: q = (coord - origin) * q_pixel."""
    pl_px = _peaks_to_pointlist(peaks_px)
    qx = (pl_px.data["qx"] - float(qx0)) * float(q_pixel)
    qy = (pl_px.data["qy"] - float(qy0)) * float(q_pixel)
    return _pointlist_from_arrays(qx, qy, pl_px.data["intensity"])


def _guess_origin_qpixel():
    qx0 = QX0_PX
    qy0 = QY0_PX
    qpx = Q_PIXEL_AINV
    # sidecar calibration if present
    if qx0 is None:
        qx0 = _read_h5_scalar(H5_PATH, "qx0_mean")
    if qy0 is None:
        qy0 = _read_h5_scalar(H5_PATH, "qy0_mean")
    if qpx is None:
        qpx = _read_h5_scalar(H5_PATH, "Q_pixel_size")
        units = _read_h5_scalar(H5_PATH, "Q_pixel_units")
        # units may be bytes; only trust numeric size if units look like A^-1
    if qx0 is None:
        qx0 = (Q_Ny - 1) / 2.0
    if qy0 is None:
        qy0 = (Q_Nx - 1) / 2.0
    return float(qx0), float(qy0), (float(qpx) if qpx is not None else None)


bragg = None
peaks_cal = None
peaks_source = None

if BRAGG_PATH is not None and Path(BRAGG_PATH).is_file():
    bragg = pl.load_braggpeaks_file(BRAGG_PATH, expected_rshape=(R_Ny, R_Nx), log=print)
    try:
        bragg.setcal()
    except Exception as exc:
        print("setcal warning:", exc)
    print("calstate:", getattr(bragg, "calstate", None))

    # Prefer calibrated peaks in Å^-1
    try:
        peaks_cal = bragg.cal[ry, rx]
        peaks_source = "bragg.cal"
    except Exception:
        peaks_cal = None

    if peaks_cal is None or len(getattr(peaks_cal, "data", peaks_cal)) == 0:
        peaks_raw = bragg.raw[ry, rx]
        qx0, qy0, qpx = _guess_origin_qpixel()
        # try calibration object
        cal = getattr(bragg, "calibration", None)
        if cal is not None and qpx is None:
            try:
                qpx = float(cal.get_Q_pixel_size())
            except Exception:
                pass
        if qpx is None:
            raise RuntimeError(
                "Picos raw disponibles pero falta Q_PIXEL_AINV (Å^-1/px). "
                "Ponlo en la celda de config."
            )
        peaks_cal = _calibrate_peaks_px_to_ainv(peaks_raw, qx0, qy0, qpx)
        peaks_source = f"bragg.raw + qx0={qx0:.3g}, qy0={qy0:.3g}, qpx={qpx:.6g}"
else:
    print("Sin BRAGG_PATH — detección local en el pixel clickeado…")
    # Probe kernel from dp_mean (sidecar) or the clicked DP itself
    probe_src = vimgs.get("dp_mean")
    if probe_src is None:
        probe_src = dp
    probe_src = np.asarray(probe_src, dtype=np.float32)
    alpha, qx0_p, qy0_p = dc.get_probe_size(probe_src)
    print(f"probe size alpha={alpha:.3g} origin=({qx0_p:.3g},{qy0_p:.3g})")

    # Build a Probe-like object via py4DSTEM helper if available
    try:
        probe = py4DSTEM.Probe.from_vacuum_image(probe_src)
        probe.get_kernel(mode="sigmoid", origin=(qx0_p, qy0_p), radii=(alpha, 2 * alpha))
        template = pl.probe_kernel_template_ndarray(probe)
    except Exception as exc:
        print("Probe.from_vacuum_image fallback:", exc)
        # simple edge-sigmoid-ish disk kernel
        yy, xx = np.indices(probe_src.shape)
        rr = np.hypot(yy - qx0_p, xx - qy0_p)
        template = np.exp(-0.5 * ((rr - alpha) / max(alpha * 0.15, 1.0)) ** 2)
        template = template - template.mean()

    detect_params = dict(
        corrPower=1.0,
        sigma=0,
        edgeBoundary=2,
        minRelativeIntensity=0.0,
        minAbsoluteIntensity=8,
        minPeakSpacing=8,
        subpixel="poly",
        maxNumPeaks=100,
    )
    # Fast4d convention for selected points: data=(rys, rxs)
    disks = dc.find_Bragg_disks(
        data=([ry], [rx]),
        template=template,
        **detect_params,
    )
    # disks may be BraggVectors (1,1) or list-like
    try:
        peaks_raw = disks.raw[0, 0]
    except Exception:
        try:
            peaks_raw = disks[0]
        except Exception:
            peaks_raw = disks

    qx0, qy0, qpx = _guess_origin_qpixel()
    if qpx is None:
        raise RuntimeError(
            "Detección OK pero falta Q_PIXEL_AINV (Å^-1/px) para ACOM. "
            "Configúralo en la celda 2 (p.ej. desde calibración Fast4d)."
        )
    # prefer probe origin if user didn't set one
    if QX0_PX is None:
        qx0 = float(qx0_p)
    if QY0_PX is None:
        qy0 = float(qy0_p)
    peaks_cal = _calibrate_peaks_px_to_ainv(peaks_raw, qx0, qy0, qpx)
    peaks_source = f"detect@({ry},{rx}) + qx0={qx0:.3g}, qy0={qy0:.3g}, qpx={qpx:.6g}"

peaks_cal = _peaks_to_pointlist(peaks_cal)
n_peaks = len(peaks_cal.data)
print(f"peaks source: {peaks_source}")
print(f"n_peaks = {n_peaks}")
if n_peaks:
    print(
        "q radius Å^-1: "
        f"min={np.hypot(peaks_cal.data['qx'], peaks_cal.data['qy']).min():.4g}  "
        f"max={np.hypot(peaks_cal.data['qx'], peaks_cal.data['qy']).max():.4g}"
    )
assert n_peaks >= 3, "ACOM necesita ≥ 3 picos calibrados en este pixel"

## 8. Visualizar picos sobre el DP (espacio detector)

Si los picos vienen de `bragg.cal` (Å⁻¹), se re-proyectan a px con origen/Q_pixel para el overlay.

In [ ]:
qx0_vis, qy0_vis, qpx_vis = _guess_origin_qpixel()
if qpx_vis is None:
    # try bragg calibration
    cal = getattr(bragg, "calibration", None) if bragg is not None else None
    if cal is not None:
        try:
            qpx_vis = float(cal.get_Q_pixel_size())
        except Exception:
            qpx_vis = None

fig_p, ax_p = plt.subplots(figsize=(5.5, 5.2), constrained_layout=True)
ax_p.imshow(E._view_filter(dp, mode=VIEW_MODE), cmap=CMAP_DP, origin="upper")

if qpx_vis is not None and qpx_vis > 0:
    qx_px = peaks_cal.data["qx"] / qpx_vis + qx0_vis
    qy_px = peaks_cal.data["qy"] / qpx_vis + qy0_vis
    ax_p.scatter(qy_px, qx_px, s=40, facecolors="none", edgecolors="cyan", linewidths=1.2)
    ax_p.set_title(f"DP + peaks (n={n_peaks})  [{peaks_source}]")
else:
    ax_p.set_title(f"DP (peaks en Å^-1; sin qpx para overlay px) n={n_peaks}")

ax_p.set_xlabel("col (qy)")
ax_p.set_ylabel("row (qx)")
plt.show()

## 9. Crystal Si + orientation plan (fiber)

- `fiber_axis = ZA_APPROX` restringe el zone axis (cono `FIBER_TILT_DEG`)
- `angle_step_in_plane` busca **0–360°** — no asume alineación con x/y

Usa el preset Si de Fast4d (`pipeline._make_crystal`).

In [ ]:
crystal = pl._make_crystal("Si")
crystal.calculate_structure_factors(float(K_MAX))
cell = getattr(crystal, "cell", None)
print(f"Si crystal cell={cell}  k_max={K_MAX}")
n_sf = None
for attr in ("g_vec", "struct_factors", "struc_factors", "g_magnitude"):
    val = getattr(crystal, attr, None)
    if val is not None:
        try:
            n_sf = len(val)
            break
        except Exception:
            pass
print(f"structure-factor beams: {n_sf}")

print(
    f"orientation_plan fiber ZA={ZA_APPROX} tilt≤{FIBER_TILT_DEG}°  "
    f"in-plane step={ANGLE_STEP_IN_PLANE}°"
)
crystal.orientation_plan(
    zone_axis_range="fiber",
    fiber_axis=list(ZA_APPROX),
    fiber_angles=[float(FIBER_TILT_DEG), 360.0],
    angle_step_zone_axis=float(ANGLE_STEP_ZONE_AXIS),
    angle_step_in_plane=float(ANGLE_STEP_IN_PLANE),
    accel_voltage=float(ACCEL_VOLTAGE),
    corr_kernel_size=float(CORR_KERNEL_SIZE),
    progress_bar=True,
)
print(
    "plan zones:",
    getattr(crystal, "orientation_num_zones", "?"),
    " in-plane steps:",
    getattr(crystal, "orientation_in_plane_steps", "?"),
)

## 10. Match ACOM en el DP clickeado + overlay `(hkl)`

`match_single_pattern` estima ZA (dentro del fiber) **y** rotación in-plane.  
`generate_diffraction_pattern` regenera picos simulados con campos `h,k,l`.

In [ ]:
orient = crystal.match_single_pattern(
    peaks_cal,
    num_matches_return=3,
    min_number_peaks=3,
    verbose=True,
    plot_corr=True,
    returnfig=False,
)

# orient may be Orientation or list/tuple of matches
if isinstance(orient, (list, tuple)):
    best = orient[0]
    print(f"returned {len(orient)} matches; using match[0]")
else:
    best = orient

for attr in ("corr", "angles", "matrix", "mirror", "inds"):
    if hasattr(best, attr):
        val = getattr(best, attr)
        try:
            print(f"{attr}:", np.asarray(val).ravel()[:6], ("..." if np.size(val) > 6 else ""))
        except Exception:
            print(f"{attr}:", val)

sim = crystal.generate_diffraction_pattern(
    best,
    sigma_excitation_error=0.03,
)
print("sim fields:", getattr(sim, "fields", sim.data.dtype.names))
print("sim n_peaks:", len(sim.data))

fig_m = py4DSTEM.process.diffraction.plot_diffraction_pattern(
    sim,
    bragg_peaks_compare=peaks_cal,
    add_labels=True,
    returnfig=True,
)
if isinstance(fig_m, tuple):
    fig_m[0].suptitle(f"Si ACOM overlay @ (ry,rx)=({ry},{rx})  ZA_approx={ZA_APPROX}")
    plt.show()
else:
    plt.show()

# tabla hkl del patrón simulado (asignación ACOM)
names = set(getattr(sim, "fields", ()) or (sim.data.dtype.names or ()))
if {"h", "k", "l"}.issubset(names):
    hkl = np.column_stack([sim.data["h"], sim.data["k"], sim.data["l"]])
    qrad = np.hypot(sim.data["qx"], sim.data["qy"])
    order = np.argsort(qrad)
    print("\nSimulated beams (sorted by |q|):")
    print(f"{'h':>4} {'k':>4} {'l':>4} {'|q|':>10} {'I':>10}")
    for i in order[:40]:
        print(
            f"{int(hkl[i,0]):4d} {int(hkl[i,1]):4d} {int(hkl[i,2]):4d} "
            f"{qrad[i]:10.4f} {float(sim.data['intensity'][i]):10.3g}"
        )
else:
    print("El PointList simulado no trae campos h,k,l en esta versión de py4DSTEM.")

## 11. Checks rápidos de consistencia (Si)

- Correlación del match (celda anterior, `verbose=True`)
- Overlay geométrico sim vs experimental
- Radios |g| de picos fuertes ≈ teoría diamond-Si tras Q-pixel correcto
- El ángulo in-plane recuperado **no** debe forzarse a 0°; compáralo con tus planos esperados

Si el match es malo: revisa `Q_PIXEL_AINV`, origen, `K_MAX`, `FIBER_TILT_DEG`, o amplía a `zone_axis_range='auto'`.

In [ ]:
# Radios experimentales vs anillos Si teóridos (solo diagnóstico)
a_si = 5.431
# reflexiones permitidas diamond (hkl all-even or all-odd); |g|=sqrt(h^2+k^2+l^2)/a
cands = []
for h in range(-6, 7):
    for k in range(-6, 7):
        for l in range(-6, 7):
            if (h, k, l) == (0, 0, 0):
                continue
            if (h % 2 == k % 2 == l % 2):
                # fcc-like parity; diamond further forbids some — enough for radius check
                cands.append(np.sqrt(h * h + k * k + l * l) / a_si)
rings = np.unique(np.round(cands, 6))
rings = rings[rings <= float(K_MAX) + 0.05]

q_exp = np.sort(np.hypot(peaks_cal.data["qx"], peaks_cal.data["qy"]))
print("Experimental |q| (first 15):", np.round(q_exp[:15], 4))
print("Si theory rings (Å^-1):     ", np.round(rings[:15], 4))

# nearest-ring residual median
if q_exp.size and rings.size:
    resid = [np.min(np.abs(rings - q)) for q in q_exp if q > 0.05]
    print(f"median |q - nearest Si ring| = {np.median(resid):.4g} Å^-1  (n={len(resid)})")